# 03 — CTF Lorenz End-to-End

Train an EML-Koopman model on the Lorenz system, extract symbolic formulas,
and evaluate using CTF metrics.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '..')

from koopman_eml import KoopmanEML
from koopman_eml.training import train_koopman_eml
from koopman_eml.analysis import (
    koopman_eigendecomposition,
    extract_eml_formulas,
    prediction_rollout,
    compute_metrics,
)
from koopman_eml.ctf import short_term_score

## 1. Generate Lorenz Data

In [ ]:
sys.path.insert(0, '../experiments/ctf_lorenz')
from generate_data import generate_lorenz_trajectories

data = generate_lorenz_trajectories(n_trajectories=20, n_steps=5000, dt=0.01)
X_k = torch.tensor(data['X_k_train'], dtype=torch.float32)
X_k1 = torch.tensor(data['X_k1_train'], dtype=torch.float32)
print(f'Training pairs: {X_k.shape}')

## 2. Train EML-Koopman

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = KoopmanEML(state_dim=3, n_observables=16, tree_depth=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

history = train_koopman_eml(
    model, X_k, X_k1,
    n_epochs=1500,
    lr=2e-3,
    device=device,
    verbose=True,
)

## 3. Training Loss Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(history['total'], label='Total')
ax1.semilogy(history['pred'], label='Prediction', alpha=0.7)
ax1.semilogy(history['recon'], label='Reconstruction', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Losses')
ax1.legend()

ax2.plot(history['tau'])
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Temperature (tau)')
ax2.set_title('Gumbel-Softmax Annealing Schedule')

plt.tight_layout()
plt.show()

## 4. Eigendecomposition & Symbolic Formulas

In [ ]:
eig = koopman_eigendecomposition(model)

print('Top Koopman eigenvalues:')
for i in range(min(10, len(eig['eigenvalues']))):
    lam = eig['eigenvalues'][i]
    print(f'  lambda_{i} = {lam:.4f}  |lambda|={abs(lam):.4f}  '
          f'freq={eig["frequencies"][i]:.4f}  growth={eig["growth_rates"][i]:.4f}')

# Eigenvalue spectrum
plt.figure(figsize=(6, 6))
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, label='Unit circle')
plt.scatter(eig['eigenvalues'].real, eig['eigenvalues'].imag, c='red', s=50, zorder=5)
plt.xlabel('Re(lambda)')
plt.ylabel('Im(lambda)')
plt.title('Koopman Eigenvalue Spectrum')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
formulas = extract_eml_formulas(model)
print('Discovered EML observables (snapped):')
for i, f in enumerate(formulas):
    print(f'  g_{i}(x) = {f}')

## 5. Forecasting Evaluation

In [ ]:
x0 = torch.tensor(data['X1train'][-1], dtype=torch.float32)
n_forecast = len(data['X1test'])
traj = prediction_rollout(model, x0, n_forecast - 1, device=device)
pred = traj[1:].numpy()
truth = data['X1test'][:len(pred)]

metrics = compute_metrics(pred, truth)
e1 = short_term_score(pred, truth)
print(f'RMSE: {metrics["rmse"]:.4f}')
print(f'Valid prediction steps: {metrics["valid_prediction_steps"]}')
print(f'CTF E1: {e1:.2f}')

In [ ]:
labels = ['x', 'y', 'z']
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
n_show = min(500, len(pred))
for i, ax in enumerate(axes):
    ax.plot(truth[:n_show, i], label=f'Truth {labels[i]}', linewidth=1.5)
    ax.plot(pred[:n_show, i], '--', label=f'Predicted {labels[i]}', linewidth=1.5)
    ax.set_ylabel(labels[i])
    ax.legend(loc='upper right')
axes[-1].set_xlabel('Time step')
axes[0].set_title('EML-Koopman Lorenz Forecast')
plt.tight_layout()
plt.show()